In [1]:

import pandas as pd
from Lmd_Utils import cleantext
from sklearn.feature_extraction.text import TfidfVectorizer
import joblib

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\lemin\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\lemin\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


['Sorry, I dont have the data to answer your question. I will forward your question to the support staff, please wait.', 0]


In [2]:
questions=[]
answers=[]

In [3]:
df=pd.read_parquet('./Data/train.parquet')
df2=pd.read_parquet('./Data/validation.parquet')

In [4]:

df.columns

Index(['id', 'uit_id', 'title', 'context', 'question', 'answers',
       'is_impossible', 'plausible_answers'],
      dtype='str')

In [5]:
df.head()

,id,uit_id,title,context,question,answers,is_impossible,plausible_answers
0,0001-0001-0001,uit_000001,Phạm Văn Đồng,Phạm Văn Đồng (1 tháng 3 năm 1906 – 29 tháng 4...,Tên gọi nào được Phạm Văn Đồng sử dụng khi làm...,"{'text': ['Lâm Bá Kiệt'], 'answer_start': [507]}",False,None
1,0001-0001-0002,uit_000002,Phạm Văn Đồng,Phạm Văn Đồng (1 tháng 3 năm 1906 – 29 tháng 4...,Phạm Văn Đồng giữ chức vụ gì trong bộ máy Nhà ...,"{'text': ['Thủ tướng'], 'answer_start': [60]}",False,None
2,0001-0001-0003,uit_000003,Phạm Văn Đồng,Phạm Văn Đồng (1 tháng 3 năm 1906 – 29 tháng 4...,"Giai đoạn năm 1955-1976, Phạm Văn Đồng nắm giữ...",{'text': ['Thủ tướng Chính phủ Việt Nam Dân ch...,False,None
3,0001-0001-0004,uit_000004,Phạm Văn Đồng,Phạm Văn Đồng (1 tháng 3 năm 1906 – 29 tháng 4...,Tên gọi nào được Phạm Văn Đồng sử dụng trước k...,"{'text': [], 'answer_start': []}",True,"{'text': ['Lâm Bá Kiệt'], 'answer_start': [507]}"
4,0001-0001-0005,uit_000005,Phạm Văn Đồng,Phạm Văn Đồng (1 tháng 3 năm 1906 – 29 tháng 4...,Hồ Học Lãm giữ chức vụ gì trong bộ máy Nhà nướ...,"{'text': [], 'answer_start': []}",True,"{'text': ['Thủ tướng'], 'answer_start': [60]}"


In [6]:
df2.head(10)

,id,uit_id,title,context,question,answers,is_impossible,plausible_answers
0,0001-0001-0001,uit_000001,Paris,Paris nằm ở điểm gặp nhau của các hành trình t...,Paris đạt được thành quả gì sau khoảng 4 thế k...,{'text': ['trở thành một trong những trung tâm...,False,None
1,0001-0001-0002,uit_000002,Paris,Paris nằm ở điểm gặp nhau của các hành trình t...,Vị trí địa lý của Pháp có gì đặc biệt?,"{'text': [], 'answer_start': []}",True,{'text': ['nằm ở điểm gặp nhau của các hành tr...
2,0001-0001-0003,uit_000003,Paris,Paris nằm ở điểm gặp nhau của các hành trình t...,Kinh tế xung quanh kinh đô ánh sáng mạnh về gì?,"{'text': ['giáo dục và nghệ thuật', 'nông nghi...",False,None
3,0001-0001-0004,uit_000004,Paris,Paris nằm ở điểm gặp nhau của các hành trình t...,Cuộc cách mạng Pháp diễn ra ở đâu?,"{'text': ['Paris'], 'answer_start': [358]}",False,None
4,0001-0001-0005,uit_000005,Paris,Paris nằm ở điểm gặp nhau của các hành trình t...,Kinh đô ánh sáng nổi tiếng về lĩnh vực gì?,"{'text': ['nghệ thuật và giải trí', 'trung tâm...",False,None
5,0001-0001-0006,uit_000006,Paris,Paris nằm ở điểm gặp nhau của các hành trình t...,Paris là một thành phố quan trọng của nước Phá...,"{'text': ['thế kỷ 10'], 'answer_start': [134]}",False,None
6,0001-0001-0007,uit_000007,Paris,Paris nằm ở điểm gặp nhau của các hành trình t...,Vị trí địa lý của Paris có gì đặc biệt?,{'text': ['nằm ở điểm gặp nhau của các hành tr...,False,None
7,0001-0002-0001,uit_000008,Paris,"Nổi tiếng với tên gọi Kinh đô ánh sáng, Paris ...",Điều gì đã nói lên Paris là thành phố lý tưởng...,"{'text': ['Kinh đô ánh sáng', 'mỗi năm có đến ...",False,None
8,0001-0002-0002,uit_000009,Paris,"Nổi tiếng với tên gọi Kinh đô ánh sáng, Paris ...",Người ta đến với Paris vì điều gì?,"{'text': ['Sự nhộn nhịp, các công trình kiến t...",False,None
9,0001-0002-0003,uit_000010,Paris,"Nổi tiếng với tên gọi Kinh đô ánh sáng, Paris ...",Điều gì đã nói lên New York là thành phố lý tư...,"{'text': [], 'answer_start': []}",True,{'text': ['mỗi năm có đến 30 triệu khách nước ...


In [7]:
df.iloc[0]

id                                                      0001-0001-0001
uit_id                                                      uit_000001
title                                                    Phạm Văn Đồng
context              Phạm Văn Đồng (1 tháng 3 năm 1906 – 29 tháng 4...
question             Tên gọi nào được Phạm Văn Đồng sử dụng khi làm...
answers               {'text': ['Lâm Bá Kiệt'], 'answer_start': [507]}
is_impossible                                                    False
plausible_answers                                                 None
Name: 0, dtype: object

In [8]:
df['question'].iloc[3]


'Tên gọi nào được Phạm Văn Đồng sử dụng trước khi làm Phó chủ nhiệm cơ quan Biện sự xứ tại Quế Lâm?'

In [9]:
df['context'].iloc[3]

'Phạm Văn Đồng (1 tháng 3 năm 1906 – 29 tháng 4 năm 2000) là Thủ tướng đầu tiên của nước Cộng hòa Xã hội chủ nghĩa Việt Nam từ năm 1976 (từ năm 1981 gọi là Chủ tịch Hội đồng Bộ trưởng) cho đến khi nghỉ hưu năm 1987. Trước đó ông từng giữ chức vụ Thủ tướng Chính phủ Việt Nam Dân chủ Cộng hòa từ năm 1955 đến năm 1976. Ông là vị Thủ tướng Việt Nam tại vị lâu nhất (1955–1987). Ông là học trò, cộng sự của Chủ tịch Hồ Chí Minh. Ông có tên gọi thân mật là Tô, đây từng là bí danh của ông. Ông còn có tên gọi là Lâm Bá Kiệt khi làm Phó chủ nhiệm cơ quan Biện sự xứ tại Quế Lâm (Chủ nhiệm là Hồ Học Lãm).'

In [10]:
df['answers'].iloc[3]

{'text': array([], dtype=object), 'answer_start': array([], dtype=int32)}

In [11]:
df['plausible_answers'].iloc[3]

{'text': array(['Lâm Bá Kiệt'], dtype=object),
 'answer_start': array([507], dtype=int32)}

In [12]:
for i in range(len(df)):
    is_imp = df['is_impossible'].iloc[i]
    ans_dict = df['answers'].iloc[i]
    plausible_dict = df['plausible_answers'].iloc[i]
    ques_text = df['question'].iloc[i]

    if is_imp == True:
        if isinstance(plausible_dict, dict) and len(plausible_dict.get('text', [])) > 0:
            answers.append(cleantext(plausible_dict['text'][0]))
            questions.append(cleantext(ques_text))
    else:
        if isinstance(ans_dict, dict) and len(ans_dict.get('text', [])) > 0:
            answers.append(cleantext(ans_dict['text'][0]))
            questions.append(cleantext(ques_text))


for i in range(len(df2)):
    is_imp = df2['is_impossible'].iloc[i]
    ans_dict = df2['answers'].iloc[i]
    plausible_dict = df2['plausible_answers'].iloc[i]
    ques_text = df2['question'].iloc[i]

    if is_imp == True:
        if isinstance(plausible_dict, dict) and len(plausible_dict.get('text', [])) > 0:
            answers.append(cleantext(plausible_dict['text'][0]))
            questions.append(cleantext(ques_text))
    else:
        if isinstance(ans_dict, dict) and len(ans_dict.get('text', [])) > 0:
            answers.append(cleantext(ans_dict['text'][0]))
            questions.append(cleantext(ques_text))

print(len(questions))
print(len(answers))
print(len(df))
print(len(df2))

32268
32268
28454
3814


In [13]:
vect=TfidfVectorizer()
X_train=vect.fit_transform(questions)
joblib.dump(vect,'./models/tfidf_QA_VN.pkl')
joblib.dump(X_train,'./models/Question_QA_VN.pkl')
joblib.dump(answers,'./models/Answer_QA_VN.pkl')


['./models/Answer_QA_VN.pkl']